---
title: | 
    Creating custom distribution in `PyMC`
author: Sheng Long 
draft: true 
---

The [old tutorial](https://www.pymc.io/projects/examples/en/stable/howto/custom_distribution.html) in the official documentation is meant for PyMC3^[... where as the latest version is `5.xx`]. The latest version of PyMC uses a different way, but we will nevertheless use the Generalized Poisson distribution as an example. 

The Generalized Poisson distribution, aka the Lagrangian Poisson distribution, models count data, and is flexible enough to handle both overdispersion and underdispersion. It has the following PMF: 

$$
p(Y = y | \theta, \lambda) = \frac{\theta(\theta+\lambda y)^{y-1}e^{-\theta-\lambda y}}{y!}
$$
where $\theta > 0$ and $\max(-1, -\frac{\theta}{4}) \leq \lambda \leq 1$

In [ ]:
import pymc as pm 
import arviz as az 
import arviz_plots as azp
import numpy as np 
from pytensor.tensor import TensorVariable, gammaln
import pytensor.tensor as pt 
from typing import Optional, Tuple

print(f'Using pymc version {pm.__version__}')

Using pymc version 5.26.1


We will first need to define the log of the probability function: 
$$
\log(f(y|\theta, \lambda)) = \log(\theta) + (y-1) \log(\theta + \lambda y) - (\theta + \lambda y) - \log(y!)
$$

In [15]:
def genpossion_logp(value: TensorVariable, theta: TensorVariable, lam: TensorVariable):
    theta_lam_value = theta + lam * value 
    log_prob = pm.math.log(theta) + (value - 1) * pm.math.log(theta_lam_value) - theta_lam_value - gammaln(value)

    log_prob = pt.switch(theta_lam_value <= 0, -pt.inf, log_prob)

     # Handle other constraints (value >= 0, mu > 0, etc.)
    log_prob = pt.switch(value < 0, -pt.inf, log_prob)
    log_prob = pt.switch(theta <= 0, -pt.inf, log_prob)
    log_prob = pt.switch(pt.abs(lam) > 1, -pt.inf, log_prob)
    log_prob = pt.switch(lam < -theta / 4, -pt.inf, log_prob)

    return log_prob 

Then define a random sample generator: 

In [19]:
def genpoisson_rvs(theta, lam, size=None):
    if size is not None:
        assert size == theta.shape
    else:
        size = theta.shape
    lam = lam[0]
    omega = np.exp(-lam)
    X = np.full(size, 0)
    S = np.exp(-theta)
    P = np.copy(S)
    for i in range(size[0]):
        U = np.random.uniform()
        while U > S[i]:
            X[i] += 1
            C = theta[i] - lam + lam * X[i]
            P[i] = omega * C * (1 + lam / C) ** (X[i] - 1) * P[i] / X[i]
            S[i] += P[i]
    return X

If you're searching for "PyMC custom distribution" online you might have encountered this class, `pymc.DensityDist`^[... [source](https://www.pymc.io/projects/docs/en/v4.4.0/api/distributions/generated/pymc.DensityDist.html) here]. Note however, that this is part of `v4` and therefore is also not the intended way to write things. 

The correct way to write things is to use the `pymc.CustomDist` class, with documentat [here](https://www.pymc.io/projects/docs/en/latest/api/distributions/custom.html). 

In [ ]:
with pm.Model() as gen_poisson_model: 
    theta = pm.Exponential("theta", 1)
    lam = pm.Uniform("lam", lower=-0.9, upper=0.9)

    y = pm.CustomDist("y", theta, lam, logp=genpossion_logp, random=genpoisson_rvs, 
    observed=observed=np.random.randn(100, 3))

SyntaxError: invalid syntax (789989737.py, line 5)

In [21]:
pm.model_to_graphviz(gen_poisson_model)

TypeError: genpoisson_rvs() got an unexpected keyword argument 'rng'
Apply node that caused the error: CustomDist_y_rv{"(),()->()"}(RNG(<Generator(PCG64) at 0x1AA0248C580>), NoneConst{None}, theta, lam)
Toposort index: 3
Inputs types: [RandomGeneratorType, <pytensor.tensor.type_other.NoneTypeT object at 0x000001AA77AC2900>, TensorType(float64, shape=()), TensorType(float64, shape=())]
Inputs shapes: ['No shapes', 'No shapes', (), ()]
Inputs strides: ['No strides', 'No strides', (), ()]
Inputs values: [Generator(PCG64) at 0x1AA0248C580, None, array(0.85235818), array(-0.14067537)]
Outputs clients: [[], [Shape(y)]]

Backtrace when the node is created (use PyTensor flag traceback__limit=N to make it longer):
  File "c:\Program Files\Positron\resources\app\extensions\positron-python\python_files\lib\ipykernel\py3\IPython\core\interactiveshell.py", line 3490, in run_ast_nodes
    if await self.run_code(code, result, async_=asy):
  File "c:\Program Files\Positron\resources\app\extensions\pos

In [ ]:


# def logp(value: TensorVariable, mu: TensorVariable) -> TensorVariable:
#     return -(value - mu)**2

# def random(
#     mu: np.ndarray | float,
#     rng: Optional[np.random.Generator] = None,
#     size : Optional[Tuple[int]]=None,
# ) -> np.ndarray | float :
#     return rng.normal(loc=mu, scale=1, size=size)

# with pm.Model():
#     mu = pm.Normal('mu', 0 , 1)
#     pm.CustomDist(
#         'custom_dist',
#         mu,
#         logp=logp,
#         random=random,
#         observed=np.random.randn(100, 3),
#         size=(100, 3),
#     )
#     prior = pm.sample_prior_predictive(10)

Sampling: [custom_dist, mu]


---
# `brms` example 

We can also use the example from [`brms`](https://cran.r-project.org/web/packages/brms/vignettes/brms_customfamilies.html) to try to create a **Beta-Binomial** model. Note that this is also implemented natively in `PyMC`^[see https://www.pymc.io/projects/docs/en/latest/api/distributions/generated/pymc.BetaBinomial.html], which allows us to double check and verify the result. 

In [8]:
%load_ext watermark
%watermark -n -u -v -iv -w -p pytensor,xarray

The watermark extension is already loaded. To reload it, use:
  %reload_ext watermark
Last updated: Fri Jan 09 2026

Python implementation: CPython
Python version       : 3.13.5
IPython version      : 8.18.1

pytensor: 2.35.1
xarray  : 2025.12.0

arviz      : 0.22.0
arviz_plots: 0.7.0
numpy      : 2.3.5
platform   : 1.0.8
pymc       : 5.26.1
pytensor   : 2.35.1
sys        : 3.13.5 (main, Jun 12 2025, 12:42:35) [MSC v.1943 64 bit (AMD64)]

Watermark: 2.5.1

